In [1]:
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam

In [2]:
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(299, 299, 3))

87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
base_model.trainable = False
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dense(1, activation='sigmoid')(x)
model = Model(inputs=base_model.input, outputs=x)

In [4]:
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [5]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    zoom_range=0.2,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    '/kaggle/input/cifake-real-and-ai-generated-synthetic-images/train',
    target_size=(299, 299),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    '/kaggle/input/cifake-real-and-ai-generated-synthetic-images/train',
    target_size=(299, 299),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

Found 80000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.


In [6]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)


model.save('real_vs_fake_inceptionv3.h5')

Epoch 1/10


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


2500/2500 ━━━━━━━━━━━━━━━━━━━━ 2740s 1s/step - accuracy: 0.8236 - loss: 0.3842 - val_accuracy: 0.8870 - val_loss: 0.2702
Epoch 2/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 2261s 902ms/step - accuracy: 0.8900 - loss: 0.2672 - val_accuracy: 0.8954 - val_loss: 0.2512
Epoch 3/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 2248s 897ms/step - accuracy: 0.8959 - loss: 0.2498 - val_accuracy: 0.9011 - val_loss: 0.2369
Epoch 4/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 2226s 888ms/step - accuracy: 0.9015 - loss: 0.2363 - val_accuracy: 0.9094 - val_loss: 0.2207
Epoch 5/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 2246s 897ms/step - accuracy: 0.9061 - loss: 0.2268 - val_accuracy: 0.9138 - val_loss: 0.2141
Epoch 6/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 2254s 900ms/step - accuracy: 0.9092 - loss: 0.2213 - val_accuracy: 0.9164 - val_loss: 0.2050
Epoch 7/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 2269s 906ms/step - accuracy: 0.9183 - loss: 0.2032 - val_accuracy: 0.9171 - val_loss: 0.2043
Epoch 8/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 2256s 901ms/step - accuracy

In [7]:
from tensorflow.keras.models import load_model


model = load_model('real_vs_fake_inceptionv3.h5')


test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    '/kaggle/input/cifake-real-and-ai-generated-synthetic-images/test',
    target_size=(299, 299),
    batch_size=32,
    class_mode='binary',
    shuffle=False  
)


loss, accuracy = model.evaluate(test_generator)
print(f"Test Accuracy: {accuracy*100:.2f}%")
print(f"Test Loss: {loss:.4f}")


Found 20000 images belonging to 2 classes.
625/625 ━━━━━━━━━━━━━━━━━━━━ 192s 296ms/step - accuracy: 0.9266 - loss: 0.1887
Test Accuracy: 93.29%
Test Loss: 0.1736
